<a href="https://colab.research.google.com/github/vivi0424/HSE-Computational-linguistics/blob/main/project_rag_llamaindex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Тема:** RAG с фреймворком LlamaIndex на основе структурированных данных

**Задачи:**
* Разработка RAG-пайплайна и сравнение результатов
1. Загрузить документы (документация Python-библиотек в формате Markdown с GitHub)
2. С помощью LlamaIndex создать индексы двух разных типов (SummaryIndex и TreeIndex)
3. Реализовать запрос к индексу с помощью SummaryIndex, найти и запустить другой ретривер
4. Интегрировать LLM для генерации ответов
5. Проанализировать простоту работы с библиотекой и возможности протестированных методов

**Библиотеки:** llama-index, huggingface, локальные модели

**Ожидаемый результат:** Colab-ноутбук с демонстрацией процесса создания и использования различных типов индексов в LlamaIndex

Выполнили: Деборина Виктория, Орлова Алина

## Установка зависимостей

In [ ]:
%pip install -q llama-index

In [ ]:
%pip install -q llama-index-llms-groq
%pip install -q llama-index-embeddings-huggingface

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-cli 0.5.6 requires llama-index-llms-openai<0.8,>=0.7.0, but you have llama-index-llms-openai 0.6.26 which is incompatible.
llama-index 0.14.18 requires llama-index-llms-openai<0.8,>=0.7.0, but you have llama-index-llms-openai 0.6.26 which is incompatible.


In [47]:
import os
from getpass import getpass
from llama_index.core import Settings
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
groq_api_key = getpass("Введите GROQ API key: ")
os.environ["GROQ_API_KEY"] = groq_api_key

llm = Groq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=2048,
)

# Эмбеддинги для retrieval
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Settings.llm = llm
Settings.embed_model = embed_model

print("LLM и embedding model настроены.")

Введите GROQ API key: ··········
LLM и embedding model настроены.


## Загрузка документов

In [ ]:
# Скачиваем репозиторий с markdown-документацией
!git clone https://github.com/github/docs.git

from pathlib import Path
import shutil

source_dir = Path("/content/docs/content")
target_dir = Path("/content/md_corpus")
target_dir.mkdir(parents=True, exist_ok=True)

md_files = list(source_dir.rglob("*.md"))
print("Найдено markdown-файлов:", len(md_files))

selected_files = md_files[:100]

for i, file_path in enumerate(selected_files, start=1):
    shutil.copy(file_path, target_dir / f"doc_{i:03d}.md")

print("Скопировано файлов:", len(list(target_dir.glob("*.md"))))

fatal: destination path 'docs' already exists and is not an empty directory.
Найдено markdown-файлов: 3522
Скопировано файлов: 120


In [ ]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("/content/md_corpus").load_data()

print("Загружено документов:", len(documents))
print("Тип первого элемента:", type(documents[0]))
print("Метаданные первого документа:", documents[0].metadata)
print("Фрагмент текста первого документа:\n")
print(documents[0].text[:1000])

Загружено документов: 120
Тип первого элемента: <class 'llama_index.core.schema.Document'>
Метаданные первого документа: {'file_path': '/content/md_corpus/doc_001.md', 'file_name': 'doc_001.md', 'file_type': 'text/markdown', 'file_size': 6071, 'creation_date': '2026-03-24', 'last_modified_date': '2026-03-24'}
Фрагмент текста первого документа:

---
title: '{% data variables.product.github %}{% ifversion fpt or ghec %}.com{% endif %} Help Documentation'
featuredLinks:
  gettingStarted:
    - /get-started/git-basics/set-up-git
    - /authentication/connecting-to-github-with-ssh
    - /repositories/creating-and-managing-repositories
    - /get-started/writing-on-github/getting-started-with-writing-and-formatting-on-github/basic-writing-and-formatting-syntax
  popular:
    - /pull-requests/collaborating-with-pull-requests/proposing-changes-to-your-work-with-pull-requests/about-pull-requests
    - /authentication
    - /copilot/how-tos/get-code-suggestions/get-ide-code-suggestions
    - /ge

## Создание узлов


In [ ]:
from llama_index.core.node_parser import MarkdownElementNodeParser

node_parser = MarkdownElementNodeParser(
    llm=llm,
    num_workers=8
)

nodes = node_parser.get_nodes_from_documents(documents)
base_nodes, objects = node_parser.get_nodes_and_objects(nodes)

print("Количество исходных узлов:", len(nodes))
print("Количество base_nodes:", len(base_nodes))
print("Количество object nodes:", len(objects))

1it [00:00, 5637.51it/s]
102it [00:00, 323370.38it/s]
1it [00:00, 9058.97it/s]
9it [00:00, 67288.30it/s]
1it [00:00, 9709.04it/s]
1it [00:00, 11881.88it/s]
1it [00:00, 12446.01it/s]
1it [00:00, 3440.77it/s]
1it [00:00, 2746.76it/s]
1it [00:00, 5899.16it/s]
11it [00:00, 120463.04it/s]
1it [00:00, 11428.62it/s]
1it [00:00, 10433.59it/s]
1it [00:00, 11214.72it/s]
1it [00:00, 4899.89it/s]
1it [00:00, 12372.58it/s]
1it [00:00, 13148.29it/s]
11it [00:00, 101178.39it/s]
15it [00:00, 170963.48it/s]
1it [00:00, 12372.58it/s]
9it [00:00, 94846.07it/s]
1it [00:00, 8305.55it/s]
1it [00:00, 8405.42it/s]
1it [00:00, 8648.05it/s]
1it [00:00, 8371.86it/s]
2it [00:00, 18196.55it/s]
1it [00:00, 11037.64it/s]
1it [00:00, 10155.70it/s]
11it [00:00, 84038.88it/s]
1it [00:00, 9664.29it/s]
5it [00:00, 47127.01it/s]
1it [00:00, 9362.29it/s]
1it [00:00, 12595.51it/s]
1it [00:00, 13066.37it/s]
1it [00:00, 9198.04it/s]
19it [00:00, 147032.80it/s]
1it [00:00, 7738.57it/s]
31it [00:00, 169965.26it/s]
21it [00:00, 

Количество исходных узлов: 266
Количество base_nodes: 230
Количество object nodes: 18


In [38]:
print("\nПример base node:\n")
print(base_nodes[0].get_content()[:1000])

if len(objects) > 0:
    print("\nПример object node:\n")
    try:
        print(objects[0].get_content()[:1000])
    except:
        print(objects[0])
else:
    print("\nObject nodes не найдены.")


Пример base node:

---
title: '{% data variables.product.github %}{% ifversion fpt or ghec %}.com{% endif %} Help Documentation'
featuredLinks:
  gettingStarted:
    - /get-started/git-basics/set-up-git
    - /authentication/connecting-to-github-with-ssh
    - /repositories/creating-and-managing-repositories
    - /get-started/writing-on-github/getting-started-with-writing-and-formatting-on-github/basic-writing-and-formatting-syntax
  popular:
    - /pull-requests/collaborating-with-pull-requests/proposing-changes-to-your-work-with-pull-requests/about-pull-requests
    - /authentication
    - /copilot/how-tos/get-code-suggestions/get-ide-code-suggestions
    - /get-started/git-basics/managing-remote-repositories
    - /pages
redirect_from:
  - /github
  - /articles
  - /common-issues-and-questions
  - /troubleshooting-common-issues
  - /early-access/github/enforcing-best-practices-with-github-policies
  - /github/enforcing-best-practices-with-github-policies/index
  - /early-access/gi

## Создание индексов



In [39]:
from llama_index.core import SummaryIndex

# Индекс по текстовым узлам
summary_index = SummaryIndex(base_nodes)

# Отдельный индекс по object nodes
summary_object_index = SummaryIndex(objects)

print("SummaryIndex создан.")
print("Количество base_nodes в индексе:", len(base_nodes))
print("Количество object nodes в индексе:", len(objects))

SummaryIndex создан.
Количество base_nodes в индексе: 230
Количество object nodes в индексе: 18


### Tree-based индексы



In [40]:
from llama_index.core import TreeIndex

# Индекс по текстовым узлам
tree_index = TreeIndex(base_nodes)

# Отдельный индекс по object nodes
tree_object_index = TreeIndex(objects)

print("TreeIndex создан.")
print("Количество base_nodes в индексе:", len(base_nodes))
print("Количество object nodes в индексе:", len(objects))

TreeIndex создан.
Количество base_nodes в индексе: 230
Количество object nodes в индексе: 18


In [41]:
query = "What information is available about GitHub Pages?"

summary_retriever = summary_index.as_retriever(
    retriever_mode="embedding",
    similarity_top_k=3
)

summary_retrieved_nodes = summary_retriever.retrieve(query)

print("SummaryIndex (embedding mode):")
print("Найдено узлов:", len(summary_retrieved_nodes))

for i, node in enumerate(summary_retrieved_nodes, 1):
    print(f"\n--- Узел {i} ---")
    try:
        print(node.get_content()[:1000])
    except:
        print(node)

SummaryIndex (embedding mode):
Найдено узлов: 3

--- Узел 1 ---
---
title: Creating a GitHub Pages site
intro: 'You can create a {% data variables.product.prodname_pages %} site in a new or existing repository.'
redirect_from:
  - /articles/creating-pages-manually
  - /articles/creating-project-pages-manually
  - /articles/creating-project-pages-from-the-command-line
  - /articles/creating-project-pages-using-the-command-line
  - /articles/creating-a-github-pages-site
  - /github/working-with-github-pages/creating-a-github-pages-site
product: '{% data reusables.gated-features.pages %}'
versions:
  fpt: '*'
  ghes: '*'
  ghec: '*'
shortTitle: Create a GitHub Pages site
category:
  - Set up a GitHub Pages site
---

 Creating a repository for your site

{% data reusables.pages.new-or-existing-repo %}

{% data reusables.repositories.create_new %}
{% data reusables.repositories.owner-drop-down %}
{% indented_data_reference reusables.pages.emu-org-only spaces=3 %}
{% data reusables.pages.cre

In [42]:
tree_retriever = tree_index.as_retriever()
tree_retrieved_nodes = tree_retriever.retrieve(query)

print("TreeIndex:")
print("Найдено узлов:", len(tree_retrieved_nodes))

for i, node in enumerate(tree_retrieved_nodes[:3], 1):
    print(f"\n--- Узел {i} ---")
    try:
        print(node.get_content()[:1000])
    except:
        print(node)

TreeIndex:
Найдено узлов: 1

--- Узел 1 ---
---
title: 'What is {% data variables.product.prodname_pages %}?'
intro: 'You can use {% data variables.product.prodname_pages %} to host a website about yourself, your organization, or your project directly from a repository on {% data variables.product.prodname_dotcom %}.'
allowTitleToDifferFromFilename: true
redirect_from:
  - /articles/what-is-github-pages
  - /articles/user-organization-and-project-pages
  - /articles/using-a-static-site-generator-other-than-jekyll
  - /articles/mime-types-on-github-pages
  - /articles/should-i-rename-usernamegithubcom-repositories-to-usernamegithubio
  - /articles/about-github-pages
  - /github/working-with-github-pages/about-github-pages
  - /early-access/github/articles/managing-your-disabled-github-pages-site
  - /pages/getting-started-with-github-pages/about-github-pages
product: '{% data reusables.gated-features.pages %}'
versions:
  fpt: '*'
  ghes: '*'
  ghec: '*'
category:
  - Learn about GitHub

In [43]:
object_query = "Show information from tables or structured summaries."

summary_object_retriever = summary_object_index.as_retriever()
summary_object_nodes = summary_object_retriever.retrieve(object_query)

print("Summary object index:")
print("Найдено узлов:", len(summary_object_nodes))

for i, node in enumerate(summary_object_nodes[:3], 1):
    print(f"\n--- Object node {i} ---")
    try:
        print(node.get_content()[:1000])
    except:
        print(node)

Summary object index:
Найдено узлов: 18

--- Object node 1 ---
Summary of articles, categories, and map topics,
with the following table title:
Content Summary,
with the following columns:
- articles: Number of articles
- categories: Number of categories
- map topics: Number of map topics

|articles	     |     31 |
|---|---|
|map topics	  |30|


--- Object node 2 ---
Recommended CodeQL CLI versions for different GitHub Enterprise Server versions,
with the following table title:
Recommended CodeQL CLI versions,
with the following columns:
- {% data variables.product.prodname_ghe_server %} version: None
- Recommended {% data variables.product.prodname_codeql_cli %} version: None

| {% data variables.product.prodname_ghe_server %} version | Recommended {% data variables.product.prodname_codeql_cli %} version |
|---|---|
|3.2| 2.23.9 ([changelog](https://codeql.github.com/docs/codeql-overview/codeql-changelog/codeql-cli-2.23.9/)) |
|3.19| 2.22.4 ([changelog](https://codeql.github.com/docs/

# Интеграция LLM для генерации ответов

### Получение узлов и синтез ответа


In [44]:
from llama_index.core.response_synthesizers import get_response_synthesizer

response_synthesizer = get_response_synthesizer(llm=llm)

summary_response = response_synthesizer.synthesize(
    query=query,
    nodes=summary_retrieved_nodes
)

tree_response = response_synthesizer.synthesize(
    query=query,
    nodes=tree_retrieved_nodes
)

print("Ответ на основе узлов SummaryIndex:\n")
print(summary_response)

print("\n" + "="*80 + "\n")

print("Ответ на основе узлов TreeIndex:\n")
print(tree_response)

Ответ на основе узлов SummaryIndex:

To get started with GitHub Pages, you can create a new repository, initialize it with a README file, and configure the repository settings. This includes choosing the repository visibility and setting up the sidebar settings. For publishing your site, you can select the deployment source and branch under the "Build and deployment" section. The publishing source can be chosen from a dropdown menu. You can also customize your site by editing the README.md file. Additionally, you can change the title and description of your site by editing the `_config.yml` file. Once your site is published, you can view it by visiting `username.github.io`, noting that changes may take up to 10 minutes to publish. GitHub Pages also provides resources for further customization and management of your site, including guides on adding content, using custom domains, and more.


Ответ на основе узлов TreeIndex:

Information available about GitHub Pages includes its definitio

### Query Engine


In [45]:
summary_query_engine = summary_index.as_query_engine(
    retriever_mode="embedding",
    similarity_top_k=3,
    llm=llm
)

tree_query_engine = tree_index.as_query_engine(
    llm=llm
)

summary_qe_response = summary_query_engine.query(query)
tree_qe_response = tree_query_engine.query(query)

print("Ответ SummaryIndex через query_engine:\n")
print(summary_qe_response)

print("\n" + "="*80 + "\n")

print("Ответ TreeIndex через query_engine:\n")
print(tree_qe_response)

Ответ SummaryIndex через query_engine:

To get started with GitHub Pages, you can create a new repository, initialize it with a README file, and choose the visibility of your repository. Once created, you can access the repository settings, navigate to the GitHub Pages section, and select the source of your deployment, such as deploying from a branch. You can also choose a publishing source from the branch dropdown menu. The README.md file in your repository is where you can write the content for your site. Additionally, you can customize your site by editing the `_config.yml` file to change the title and description of your site. GitHub Pages allows you to publish any static files that are pushed to the repository, making it a great way to showcase open source projects, host a blog, or share a résumé. You can view your new website by visiting `username.github.io`, and it may take up to 10 minutes for changes to your site to publish after you push the changes to GitHub.


Ответ TreeInd

## Анализ библиотеки, сравнение методов


Для сравнения `SummaryIndex` и `TreeIndex` были рассмотрены три типа запросов:

**Фактический вопрос:** *What information is available about GitHub Pages?*

На этот вопрос оба индекса дали корректные ответы, но с разным характером.
`SummaryIndex` вернул более подробный и прикладной ответ, включающий шаги по созданию сайта, настройке репозитория и публикации.
`TreeIndex`, напротив, дал более короткий и обобщенный ответ, описывающий, что такое GitHub Pages и для каких задач он используется.

**Обобщающий вопрос:** *How can a GitHub Pages site be created?*

По уже полученным ответам видно, что `SummaryIndex` лучше подходит для процедурных и пошаговых запросов, так как извлекает несколько релевантных узлов и на их основе формирует более детализированное описание процесса.

**Уточняющий вопрос к структурированным данным**: *What structured information is present in the documentation tables?*

Для такого типа запроса полезны `object nodes`, полученные через `MarkdownElementNodeParser`.
В них были выделены таблицы со структурированной информацией, например:

*   Recommended CodeQL CLI versions for different GitHub Enterprise Server versions
*   Minimum Runner versions for different GitHub Enterprise Server versions

## Выводы

### SummaryIndex vs TreeIndex

| Критерий | SummaryIndex | TreeIndex |
|---|---|---|
| Скорость создания | Проще и быстрее | Медленнее, так как используется <br> иерархическая суммаризация |
| Поведение при поиске | Хорошо подходит для поиска <br> по релевантным узлам <br> и последующей суммаризации | Хорошо подходит для иерархической <br> организации информации |
| Характер ответа | Более подробный, <br> с большим количеством контекста | Более компактный и обобщенный |
| Подходит для | Общих и процедурных вопросов | Навигационных и уточняющих вопросов |
| Зависимость от LLM | Используется при генерации ответа | Используется и при построении, и при генерации ответа |

### Сильные стороны LlamaIndex

Эксперимент показал, что `LlamaIndex` удобно использовать для работы со структурированными markdown-документами. Библиотека позволяет:

* загружать и хранить документы вместе с метаданными;
* преобразовывать markdown в текстовые и объектные узлы;
* строить разные типы индексов для одних и тех же данных;
* использовать LLM как на этапе структурирования данных, так и на этапе генерации ответа.

Это делает LlamaIndex удобным инструментом для задач RAG, особенно когда документы содержат не только обычный текст, но и таблицы и другие структурированные элементы.